# 기자명 채우기

1. 아래 셀을 실행하고, 파일 선택 창에서 **랭킹·발행 엑셀을 모두** 선택하세요.
2. 끝나면 `reporters.csv` 가 내려받아집니다. 그 파일을 Claude 대화창에 올려주세요.

기사 1만 건 기준 15~20분 정도 걸립니다.

In [ ]:
# 기자명 수집기 – 엑셀 링크를 모두 방문해 기자명·입력시각을 읽어 CSV로 저장합니다.
# 실행하면 파일 선택 창이 뜹니다 → 가지고 계신 랭킹/발행 엑셀을 모두 선택하세요(여러 개 가능).
!pip -q install beautifulsoup4 openpyxl
import re, csv, time, io
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from bs4 import BeautifulSoup
from openpyxl import load_workbook
from google.colab import files

up = files.upload()
keys = {}
for name, data in up.items():
    wb = load_workbook(io.BytesIO(data), read_only=True, data_only=True)
    for ws in wb.worksheets:
        for row in ws.iter_rows(values_only=True):
            for v in row:
                m = re.search(r"/article/(?:\w+/)?(\d{3})/(\d{6,})", str(v or ""))
                if m:
                    keys[(m.group(1), m.group(2))] = 1
keys = list(keys)
print(f"고유 기사 {len(keys)}건 확인 시작 (약 {len(keys)//600+1}분)")

S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
                  "Accept-Language": "ko-KR,ko;q=0.9", "Referer": "https://news.naver.com/"})
S.mount("https://", requests.adapters.HTTPAdapter(pool_maxsize=16, max_retries=3))

def norm_dt(raw):
    m = re.search(r"(\d{4})[-.](\d{2})[-.](\d{2})[ T]?(\d{2}):(\d{2})", raw or "")
    return f"{m[1]}-{m[2]}-{m[3]} {m[4]}:{m[5]}" if m else ""

def fetch(oid, aid):
    r = S.get(f"https://n.news.naver.com/mnews/article/{oid}/{aid}", timeout=12)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    t = soup.select_one("span.media_end_head_info_datestamp_time[data-date-time]")
    pub = norm_dt(t.get("data-date-time")) if t else ""
    if not pub:
        mt = soup.select_one('meta[property="article:published_time"]')
        pub = norm_dt(mt.get("content") if mt else "")
    names = []
    for tag in soup.select("em.media_end_head_journalist_name, span.byline_s, .byline .byline_s, .media_end_head_journalist_box em"):
        n = re.sub(r"\(.*?\)|[\w.+-]+@[\w.-]+", "", tag.get_text(" ", strip=True))
        n = re.sub(r"\s*(기자|특파원|객원기자|선임기자|전문기자|논설위원|위원|칼럼니스트).*$", "", n).strip()
        if n and n not in names and len(n) <= 20:
            names.append(n)
    return pub, "·".join(names[:3])

out, fails = [], 0
with ThreadPoolExecutor(8) as ex:
    futs = {ex.submit(fetch, *k): k for k in keys}
    for i, f in enumerate(as_completed(futs), 1):
        oid, aid = futs[f]
        try:
            pub, rep = f.result()
            out.append([oid, aid, pub, rep])
        except Exception:
            fails += 1
        if i % 500 == 0:
            print(f"  {i}/{len(keys)}")

with open("reporters.csv", "w", newline="", encoding="utf-8-sig") as fp:
    w = csv.writer(fp); w.writerow(["oid", "aid", "입력시각", "기자"]); w.writerows(out)
got = sum(1 for r in out if r[3])
print(f"✅ 완료: {len(out)}건 중 기자명 {got}건 확인, 실패 {fails}건")
files.download("reporters.csv")